In [6]:
!pip install -q -U gymnasium stable-baselines3 tqdm rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 23.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.11.1 requires rich<15.0.0,>=10.11.0, but you have rich 15.0.0 which is incompatible.
bigframes 2.42.0 requires rich<14,>=12.4.4, but you have rich 15.0.0 which is incompatible.


In [7]:
import os
import random
import warnings

import gymnasium as gym
from gymnasium import spaces

import numpy as np
import pandas as pd

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.utils import set_random_seed

warnings.filterwarnings("ignore", category=DeprecationWarning)

print("Libraries imported successfully.")

Libraries imported successfully.


In [8]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
set_random_seed(SEED)

print("Random seed:", SEED)

Random seed: 42


In [9]:
DATA_PATH = "processed_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset loaded successfully.
Dataset shape: (3323, 15)

Columns:
['Crop', 'Crop_Year', 'Season', 'State', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', 'Yield', 'Commodity', 'Modal_Price', 'Revenue', 'Cost', 'Profit']


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield,Commodity,Modal_Price,Revenue,Cost,Profit
0,Cotton,1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909,Cotton,7018.952055,5.573048e+06,166039.72,5.407008e+06
1,Maize,1997,Kharif,Assam,19216.0,14721,2051.4,1828786.72,5956.96,0.615652,Maize,2600.000000,3.827460e+07,1834743.68,3.643986e+07
2,Onion,1997,Whole Year,Assam,7832.0,17943,2051.4,745371.44,2427.92,2.342609,Onion,1464.607920,2.627946e+07,747799.36,2.553166e+07
3,Potato,1997,Whole Year,Assam,75259.0,671871,2051.4,7162399.03,23330.29,7.561304,Potato,931.947187,6.261483e+08,7185729.32,6.189626e+08
4,Wheat,1997,Rabi,Assam,84698.0,110054,2051.4,8060708.66,26256.38,1.259524,Wheat,2462.426606,2.709999e+08,8086965.04,2.629129e+08


In [10]:
required_columns = [
    "Crop",
    "Annual_Rainfall",
    "Modal_Price",
    "Fertilizer",
    "Pesticide",
    "Yield",
    "Profit"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("All required columns are available.")

All required columns are available.


In [11]:
training_columns = [
    "Crop",
    "Annual_Rainfall",
    "Modal_Price",
    "Fertilizer",
    "Pesticide",
    "Yield",
    "Profit"
]

df = df[training_columns].copy()

df["Crop"] = (
    df["Crop"]
    .astype(str)
    .str.strip()
    .str.lower()
)

numeric_columns = [
    "Annual_Rainfall",
    "Modal_Price",
    "Fertilizer",
    "Pesticide",
    "Yield",
    "Profit"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df = df.replace([np.inf, -np.inf], np.nan)

df = df.dropna(
    subset=["Crop", *numeric_columns]
).reset_index(drop=True)

df = df[df["Crop"] != ""].reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
print("Number of crops:", df["Crop"].nunique())
print("\nAvailable crops:")
print(df["Crop"].unique())

Cleaned dataset shape: (3323, 7)
Number of crops: 6

Available crops:
['cotton' 'maize' 'onion' 'potato' 'wheat' 'banana']


In [12]:
crop_counts = (
    df["Crop"]
    .value_counts()
    .sort_values(ascending=False)
)

print(crop_counts)

Crop
maize     975
potato    628
wheat     545
cotton    476
onion     454
banana    245
Name: count, dtype: int64


In [13]:
class FarmEnv(gym.Env):
    """
    Custom Gymnasium environment for long-term crop planning.

    State:
        1. Annual rainfall
        2. Market price
        3. Fertilizer usage
        4. Pesticide usage
        5. Crop yield
        6. Farmer savings

    Action:
        Select one crop from the available crop list.

    Reward:
        Normalised profit generated by the selected crop.

    Episode:
        A configurable planning horizon, for example 30 years.
    """

    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        data: pd.DataFrame,
        episode_length: int = 30,
        initial_savings: float = 100000.0
    ):
        super().__init__()

        if data.empty:
            raise ValueError("The environment dataset is empty.")

        self.df = data.reset_index(drop=True).copy()

        self.episode_length = int(episode_length)
        self.initial_savings = float(initial_savings)

        # Available crops
        self.crops = sorted(self.df["Crop"].unique().tolist())

        if len(self.crops) < 2:
            raise ValueError(
                "At least two different crops are required."
            )

        # Store each crop's records separately
        self.crop_data = {
            crop: self.df[self.df["Crop"] == crop]
            .reset_index(drop=True)
            for crop in self.crops
        }

        # One discrete action for each crop
        self.action_space = spaces.Discrete(len(self.crops))

        # Values used to normalise observations
        self.rainfall_scale = max(
            float(self.df["Annual_Rainfall"].abs().max()),
            1.0
        )

        self.price_scale = max(
            float(self.df["Modal_Price"].abs().max()),
            1.0
        )

        self.fertilizer_scale = max(
            float(self.df["Fertilizer"].abs().max()),
            1.0
        )

        self.pesticide_scale = max(
            float(self.df["Pesticide"].abs().max()),
            1.0
        )

        self.yield_scale = max(
            float(self.df["Yield"].abs().max()),
            1.0
        )

        self.profit_scale = max(
            float(self.df["Profit"].abs().quantile(0.95)),
            1.0
        )

        self.savings_scale = max(
            self.initial_savings,
            self.profit_scale * self.episode_length
        )

        # Normalised observations
        self.observation_space = spaces.Box(
            low=np.full(6, -10.0, dtype=np.float32),
            high=np.full(6, 10.0, dtype=np.float32),
            dtype=np.float32
        )

        self.current_step = 0
        self.savings = self.initial_savings
        self.current_scenario_index = 0
        self.last_info = {}

    def _get_scenario_row(self):
        """
        Return the current environmental scenario row.
        """
        index = self.current_scenario_index % len(self.df)
        return self.df.iloc[index]

    def _get_crop_row(self, selected_crop):
        """
        Return a record belonging to the crop selected by the agent.
        """
        crop_records = self.crop_data[selected_crop]

        index = self.current_step % len(crop_records)

        return crop_records.iloc[index]

    def _get_observation(self, row):
        """
        Build and normalise the six-element observation.
        """
        observation = np.array(
            [
                row["Annual_Rainfall"] / self.rainfall_scale,
                row["Modal_Price"] / self.price_scale,
                row["Fertilizer"] / self.fertilizer_scale,
                row["Pesticide"] / self.pesticide_scale,
                row["Yield"] / self.yield_scale,
                self.savings / self.savings_scale
            ],
            dtype=np.float32
        )

        return np.clip(
            observation,
            self.observation_space.low,
            self.observation_space.high
        ).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.current_step = 0
        self.savings = self.initial_savings

        # Start from a random dataset location during training
        self.current_scenario_index = int(
            self.np_random.integers(0, len(self.df))
        )

        scenario_row = self._get_scenario_row()
        observation = self._get_observation(scenario_row)

        info = {
            "step": self.current_step,
            "savings": self.savings
        }

        return observation, info

    def step(self, action):
        action = int(action)

        if not self.action_space.contains(action):
            raise ValueError(f"Invalid action: {action}")

        selected_crop = self.crops[action]

        scenario_row = self._get_scenario_row()
        crop_row = self._get_crop_row(selected_crop)

        scenario_rainfall = float(
            scenario_row["Annual_Rainfall"]
        )

        crop_expected_rainfall = float(
            self.crop_data[selected_crop][
                "Annual_Rainfall"
            ].mean()
        )

        # Rainfall adjustment:
        # poor rainfall suitability reduces crop profit
        rainfall_ratio = scenario_rainfall / max(
            crop_expected_rainfall,
            1.0
        )

        rainfall_factor = float(
            np.clip(rainfall_ratio, 0.50, 1.25)
        )

        base_profit = float(crop_row["Profit"])

        adjusted_profit = base_profit * rainfall_factor

        # Raw financial update
        self.savings += adjusted_profit

        # PPO receives a scaled reward for stable training
        reward = adjusted_profit / self.profit_scale

        self.current_step += 1
        self.current_scenario_index += 1

        terminated = self.current_step >= self.episode_length
        truncated = False

        if terminated:
            next_observation = np.zeros(
                6,
                dtype=np.float32
            )
        else:
            next_scenario_row = self._get_scenario_row()
            next_observation = self._get_observation(
                next_scenario_row
            )

        self.last_info = {
            "step": self.current_step,
            "selected_crop": selected_crop,
            "base_profit": base_profit,
            "rainfall_factor": rainfall_factor,
            "adjusted_profit": adjusted_profit,
            "savings": self.savings,
            "raw_reward": adjusted_profit
        }

        return (
            next_observation,
            float(reward),
            terminated,
            truncated,
            self.last_info
        )

    def render(self):
        if not self.last_info:
            print("Environment has not taken a step yet.")
            return

        print(
            f"Step: {self.last_info['step']} | "
            f"Crop: {self.last_info['selected_crop']} | "
            f"Profit: {self.last_info['adjusted_profit']:.2f} | "
            f"Savings: {self.last_info['savings']:.2f}"
        )

In [14]:
env = FarmEnv(
    data=df,
    episode_length=30,
    initial_savings=100000
)

print("Environment created successfully.")
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Number of available actions:", env.action_space.n)

print("\nAction mapping:")

for action_number, crop in enumerate(env.crops):
    print(action_number, "=", crop)

Environment created successfully.
Observation space: Box(-10.0, 10.0, (6,), float32)
Action space: Discrete(6)
Number of available actions: 6

Action mapping:
0 = banana
1 = cotton
2 = maize
3 = onion
4 = potato
5 = wheat


In [15]:
check_env(
    env,
    warn=True,
    skip_render_check=True
)

print("FarmEnv passed the Stable-Baselines3 environment check.")

FarmEnv passed the Stable-Baselines3 environment check.


In [16]:
observation, info = env.reset(seed=SEED)

print("Initial observation:")
print(observation)

print("\nRandom action test:")

for step_number in range(5):
    action = env.action_space.sample()

    (
        observation,
        reward,
        terminated,
        truncated,
        info
    ) = env.step(action)

    print("-" * 60)
    print("Step:", step_number + 1)
    print("Action:", action)
    print("Selected crop:", info["selected_crop"])
    print("Base profit:", round(info["base_profit"], 2))
    print(
        "Rainfall factor:",
        round(info["rainfall_factor"], 3)
    )
    print(
        "Adjusted profit:",
        round(info["adjusted_profit"], 2)
    )
    print("Scaled PPO reward:", round(reward, 4))
    print("Savings:", round(info["savings"], 2))

    if terminated or truncated:
        print("Episode finished.")
        break

Initial observation:
[2.7086544e-01 3.5082540e-01 4.3356027e-03 3.4671095e-03 1.1460540e-03
 2.2997031e-07]

Random action test:
------------------------------------------------------------
Step: 1
Action: 2
Selected crop: maize
Base profit: 36439856.32
Rainfall factor: 1.25
Adjusted profit: 45549820.4
Scaled PPO reward: 0.0031
Savings: 45649820.4
------------------------------------------------------------
Step: 2
Action: 5
Selected crop: wheat
Base profit: 267890456.83
Rainfall factor: 0.79
Adjusted profit: 211589327.09
Scaled PPO reward: 0.0146
Savings: 257239147.49
------------------------------------------------------------
Step: 3
Action: 3
Selected crop: onion
Base profit: 26061429.38
Rainfall factor: 0.939
Adjusted profit: 24479774.03
Scaled PPO reward: 0.0017
Savings: 281718921.52
------------------------------------------------------------
Step: 4
Action: 2
Selected crop: maize
Base profit: 53392779.6
Rainfall factor: 0.814
Adjusted profit: 43443537.6
Scaled PPO reward: 0.003

In [17]:
BASE_OUTPUT_DIR = "/content/ppo_outputs"
MODEL_DIR = os.path.join(BASE_OUTPUT_DIR, "models")
LOG_DIR = os.path.join(BASE_OUTPUT_DIR, "logs")
CHECKPOINT_DIR = os.path.join(
    BASE_OUTPUT_DIR,
    "checkpoints"
)

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Output folders created:")
print("Model directory:", MODEL_DIR)
print("Log directory:", LOG_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)

Output folders created:
Model directory: /content/ppo_outputs/models
Log directory: /content/ppo_outputs/logs
Checkpoint directory: /content/ppo_outputs/checkpoints


In [18]:
train_env = FarmEnv(
    data=df,
    episode_length=30,
    initial_savings=100000
)

train_env = Monitor(
    train_env,
    filename=os.path.join(
        LOG_DIR,
        "ppo_training_monitor.csv"
    ),
    info_keywords=(
        "savings",
        "selected_crop",
        "adjusted_profit"
    )
)

print("Monitored training environment created.")

Monitored training environment created.


In [19]:
checkpoint_callback = CheckpointCallback(
    save_freq=10000,
    save_path=CHECKPOINT_DIR,
    name_prefix="ppo_farm_checkpoint",
    save_replay_buffer=False,
    save_vecnormalize=False
)

print("Checkpoint callback created.")

Checkpoint callback created.


In [20]:
model = PPO(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=3e-4,
    n_steps=600,
    batch_size=60,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.20,
    ent_coef=0.01,
    vf_coef=0.50,
    max_grad_norm=0.50,
    verbose=1,
    tensorboard_log=LOG_DIR,
    seed=SEED,
    device="auto"
)

print("PPO model created successfully.")

Using cpu device
Wrapping the env in a DummyVecEnv.
PPO model created successfully.


In [21]:
print("PPO configuration")
print("-" * 40)
print("Policy:", model.policy_class.__name__)
print("Learning rate:", model.learning_rate)
print("Steps per rollout:", model.n_steps)
print("Batch size:", model.batch_size)
print("Training epochs:", model.n_epochs)
print("Discount factor:", model.gamma)
print("GAE lambda:", model.gae_lambda)
print("Clip range:", model.clip_range)
print("Device:", model.device)

PPO configuration
----------------------------------------
Policy: ActorCriticPolicy
Learning rate: 0.0003
Steps per rollout: 600
Batch size: 60
Training epochs: 10
Discount factor: 0.99
GAE lambda: 0.95
Clip range: FloatSchedule(ConstantSchedule(val=0.2))
Device: cpu


In [22]:
TOTAL_TIMESTEPS = 100_000

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_callback,
    progress_bar=True,
    reset_num_timesteps=True
)

print("PPO training completed.")

Logging to /content/ppo_outputs/logs/PPO_1


Output()

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 30       |
|    ep_rew_mean     | 1.77     |
| time/              |          |
|    fps             | 507      |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 600      |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 30          |
|    ep_rew_mean          | 1.91        |
| time/                   |             |
|    fps                  | 500         |
|    iterations           | 2           |
|    time_elapsed         | 2           |
|    total_timesteps      | 1200        |
| train/                  |             |
|    approx_kl            | 0.009475179 |
|    clip_fraction        | 0.034       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.79       |
|    explained_variance   | 0.0157      |
|    learning_rate        | 0.

PPO training completed.


In [23]:
FINAL_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "ppo_farmer_crop_planning"
)

model.save(FINAL_MODEL_PATH)

print("Final PPO model saved.")
print("Path:", FINAL_MODEL_PATH + ".zip")

Final PPO model saved.
Path: /content/ppo_outputs/models/ppo_farmer_crop_planning.zip


In [24]:
train_env.close()

print("Training environment closed.")

Training environment closed.


In [25]:
loaded_model = PPO.load(
    FINAL_MODEL_PATH,
    device="auto"
)

print("Saved PPO model loaded successfully.")

Saved PPO model loaded successfully.


In [26]:
evaluation_env = FarmEnv(
    data=df,
    episode_length=30,
    initial_savings=100000
)

evaluation_env = Monitor(evaluation_env)

print("Evaluation environment created.")

Evaluation environment created.


In [27]:
mean_reward, std_reward = evaluate_policy(
    loaded_model,
    evaluation_env,
    n_eval_episodes=10,
    deterministic=True
)

print("Basic PPO evaluation")
print("-" * 40)
print("Mean scaled reward:", round(mean_reward, 4))
print("Reward standard deviation:", round(std_reward, 4))

Basic PPO evaluation
----------------------------------------
Mean scaled reward: 2.8002
Reward standard deviation: 0.5389


In [28]:
test_env = FarmEnv(
    data=df,
    episode_length=30,
    initial_savings=100000
)

observation, info = test_env.reset(seed=SEED)

episode_results = []
total_scaled_reward = 0.0

while True:
    action, _ = loaded_model.predict(
        observation,
        deterministic=True
    )

    (
        observation,
        reward,
        terminated,
        truncated,
        info
    ) = test_env.step(action)

    total_scaled_reward += float(reward)

    episode_results.append(
        {
            "Year": info["step"],
            "Action": int(action),
            "Selected_Crop": info["selected_crop"],
            "Base_Profit": info["base_profit"],
            "Rainfall_Factor": info["rainfall_factor"],
            "Adjusted_Profit": info["adjusted_profit"],
            "Savings": info["savings"],
            "Scaled_Reward": reward
        }
    )

    if terminated or truncated:
        break

ppo_results_df = pd.DataFrame(episode_results)

print("30-year test completed.")
print("Total scaled reward:", round(total_scaled_reward, 4))
print(
    "Final savings:",
    round(ppo_results_df["Savings"].iloc[-1], 2)
)

ppo_results_df.head(10)

30-year test completed.
Total scaled reward: 3.3168
Final savings: 48075275459.84


,Year,Action,Selected_Crop,Base_Profit,Rainfall_Factor,Adjusted_Profit,Savings,Scaled_Reward
0,1,1,cotton,5.407008e+06,1.225687,6.627302e+06,6.727302e+06,0.000457
1,2,1,cotton,3.851564e+07,0.791597,3.048886e+07,3.721616e+07,0.002103
2,3,1,cotton,3.625030e+08,0.791597,2.869563e+08,3.241725e+08,0.019797
3,4,1,cotton,2.566162e+06,0.791597,2.031366e+06,3.262038e+08,0.000140
4,5,1,cotton,5.107536e+06,0.791597,4.043110e+06,3.302469e+08,0.000279
5,6,1,cotton,6.793576e+09,0.791597,5.377774e+09,5.708021e+09,0.371018
6,7,1,cotton,1.235105e+08,0.791597,9.777057e+07,5.805791e+09,0.006745
7,8,1,cotton,3.855108e+07,0.791597,3.051692e+07,5.836308e+09,0.002105
8,9,1,cotton,4.925655e+05,0.791597,3.899133e+05,5.836698e+09,0.000027
9,10,1,cotton,5.061961e+06,1.250000,6.327452e+06,5.843025e+09,0.000437


In [29]:
print("Crop-selection frequency:")

crop_frequency = (
    ppo_results_df["Selected_Crop"]
    .value_counts()
)

print(crop_frequency)

Crop-selection frequency:
Selected_Crop
cotton    30
Name: count, dtype: int64


In [30]:
RESULTS_PATH = os.path.join(
    BASE_OUTPUT_DIR,
    "ppo_basic_30_year_test.csv"
)

ppo_results_df.to_csv(
    RESULTS_PATH,
    index=False
)

print("Basic PPO test results saved.")
print("Path:", RESULTS_PATH)

Basic PPO test results saved.
Path: /content/ppo_outputs/ppo_basic_30_year_test.csv


In [31]:
print("Phase 3 outputs")
print("=" * 50)

for root, directories, files in os.walk(
    BASE_OUTPUT_DIR
):
    level = root.replace(
        BASE_OUTPUT_DIR,
        ""
    ).count(os.sep)

    indent = "    " * level

    print(
        f"{indent}{os.path.basename(root)}/"
    )

    sub_indent = "    " * (level + 1)

    for filename in files:
        print(f"{sub_indent}{filename}")

Phase 3 outputs
ppo_outputs/
    ppo_basic_30_year_test.csv
    checkpoints/
        ppo_farm_checkpoint_50000_steps.zip
        ppo_farm_checkpoint_90000_steps.zip
        ppo_farm_checkpoint_70000_steps.zip
        ppo_farm_checkpoint_10000_steps.zip
        ppo_farm_checkpoint_60000_steps.zip
        ppo_farm_checkpoint_80000_steps.zip
        ppo_farm_checkpoint_40000_steps.zip
        ppo_farm_checkpoint_100000_steps.zip
        ppo_farm_checkpoint_30000_steps.zip
        ppo_farm_checkpoint_20000_steps.zip
    logs/
        ppo_training_monitor.csv
        PPO_1/
            events.out.tfevents.1784301593.d65360412472.765.0
    models/
        ppo_farmer_crop_planning.zip


In [32]:
print("Phase 3 outputs")
print("=" * 50)

for root, directories, files in os.walk(
    BASE_OUTPUT_DIR
):
    level = root.replace(
        BASE_OUTPUT_DIR,
        ""
    ).count(os.sep)

    indent = "    " * level

    print(
        f"{indent}{os.path.basename(root)}/"
    )

    sub_indent = "    " * (level + 1)

    for filename in files:
        print(f"{sub_indent}{filename}")

Phase 3 outputs
ppo_outputs/
    ppo_basic_30_year_test.csv
    checkpoints/
        ppo_farm_checkpoint_50000_steps.zip
        ppo_farm_checkpoint_90000_steps.zip
        ppo_farm_checkpoint_70000_steps.zip
        ppo_farm_checkpoint_10000_steps.zip
        ppo_farm_checkpoint_60000_steps.zip
        ppo_farm_checkpoint_80000_steps.zip
        ppo_farm_checkpoint_40000_steps.zip
        ppo_farm_checkpoint_100000_steps.zip
        ppo_farm_checkpoint_30000_steps.zip
        ppo_farm_checkpoint_20000_steps.zip
    logs/
        ppo_training_monitor.csv
        PPO_1/
            events.out.tfevents.1784301593.d65360412472.765.0
    models/
        ppo_farmer_crop_planning.zip


In [35]:
import shutil
from google.colab import files

# Create a zip archive of the entire BASE_OUTPUT_DIR
shutil.make_archive(
    base_name="/content/ppo_outputs",
    format="zip",
    root_dir="/content",
    base_dir="ppo_outputs"
)

files.download("/content/ppo_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>